<a href="https://colab.research.google.com/github/nighttmarionne/Test-1/blob/main/VOVAHACK1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error

In [ ]:
#vova file

In [ ]:
url = (
    "https://raw.githubusercontent.com/"
    "mwaskom/seaborn-data/master/mpg.csv"
)

cars = pd.read_csv(url)

cars = cars.dropna(subset=["mpg"])

features = ["weight", "horsepower", "cylinders"]

X = cars[features]
y = cars["mpg"]

print("Cars loaded:", len(cars))
print("\nFirst five cars:")
print(cars[features + ["mpg"]].head())

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training cars:", len(X_train))
print("Testing cars:", len(X_test))

Training cars: 318
Testing cars: 80


In [4]:
model = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestRegressor(
        n_estimators=200,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    )
)

model.fit(X_train, y_train)

print("Training complete!")

Training complete!


In [5]:
predictions = model.predict(X_test)

model_error = mean_absolute_error(
    y_test,
    predictions
)

baseline = DummyRegressor(strategy="mean")

baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_test)

baseline_error = mean_absolute_error(
    y_test,
    baseline_predictions
)

print("\nAI average test error:", round(model_error, 2), "MPG")
print("Basic method's test error:", round(baseline_error, 2), "MPG")

print("\nExample test results:")

results = pd.DataFrame({
    "Actual MPG": y_test.to_numpy(),
    "Predicted MPG": predictions.round(1)
})

print(results.head().to_string(index=False))


AI average test error: 2.49 MPG
Basic method's test error: 5.96 MPG

Example test results:
 Actual MPG  Predicted MPG
       33.0           31.7
       28.0           26.9
       19.0           21.5
       13.0           14.2
       14.0           15.0


In [6]:
def predict_car(weight_kg, horsepower, cylinders):

    values = [weight_kg, horsepower, cylinders]

    if not all(np.isfinite(value) for value in values):
        raise ValueError("Use finite numbers.")

    if weight_kg <= 0 or horsepower <= 0:
        raise ValueError("Weight and horsepower must be positive.")

    if not float(cylinders).is_integer():
        raise ValueError("Cylinders must be a whole number.")

    known_cylinders = set(X_train["cylinders"].unique())

    if cylinders not in known_cylinders:
        allowed = sorted(int(value) for value in known_cylinders)
        raise ValueError(
            f"Use a cylinder count represented in training: {allowed}"
        )

    weight_lb = weight_kg * 2.2046226218

    new_car = pd.DataFrame(
        [[weight_lb, horsepower, cylinders]],
        columns=features
    )

    for feature in ["weight", "horsepower"]:
        value = new_car.loc[0, feature]
        minimum = X_train[feature].min()
        maximum = X_train[feature].max()

        if value < minimum or value > maximum:
            print(
                f"Warning: {feature} is outside the training range. "
                "This prediction may be unreliable."
            )

    predicted_mpg = float(model.predict(new_car)[0])

    litres_per_100km = 235.214583 / predicted_mpg

    return predicted_mpg, litres_per_100km

In [ ]:
print("Fuel-efficiency AI is ready!")
print("Enter: weight in kg, horsepower, cylinders")
print("Example: 1400, 120, 4")
print("Type 'quit' to stop.")
print("Historical car dataset; not designed for EVs.")

while True:

    user_input = input("\nCar measurements: ").strip()

    if user_input.lower() == "quit":
        print("Goodbye!")
        break

    try:
        values = [
            float(part.strip())
            for part in user_input.split(",")
        ]

        if len(values) != 3:
            raise ValueError("Enter exactly three numbers.")

        weight_kg, horsepower, cylinders = values

        mpg, fuel_use = predict_car(
            weight_kg,
            horsepower,
            cylinders
        )

        print(f"Predicted fuel efficiency: {mpg:.1f} MPG")
        print(f"Equivalent fuel consumption: {fuel_use:.1f} L/100 km")
        print("This is an estimate, not a guaranteed result.")

    except ValueError as error:
        print("Input error:", error)

Fuel-efficiency AI is ready!
Enter: weight in kg, horsepower, cylinders
Example: 1400, 120, 4
Type 'quit' to stop.
Historical car dataset; not designed for EVs.

Car measurements: 0823 4234
Input error: could not convert string to float: '0823 4234'
